In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
import sunpy.visualization.colormaps as cm

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files_hmi = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

data_hmi = []
for file in files_hmi:
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data_hmi.append(data)

data_hmi = np.array(data_hmi)

In [3]:
files_fdt = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))

data_fdt = []
for file in files_fdt:
    s = np.load(file)
    mean = s['mean']
    data_fdt.append(mean)

data_fdt = np.array(data_fdt)

In [4]:
files_fdt[11]

np.str_('/home/ulyanov/data/solo/phi/synoptic_maps/2290.npz')

In [5]:
fig, ax = plt.subplots(figsize=(14,8))
ax.imshow(data_fdt[11], aspect='auto', origin='lower', cmap='hmimag',
           extent=(0, 360, -1, 1), vmin=-1000, vmax=1000)
ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))
plt.tight_layout()

In [54]:
flux_hmi = np.nanmean(data_hmi, axis=-1)
flux_fdt = np.nanmean(data_fdt, axis=-1)

dsine = 1 / 359.5
lat = np.arange(-1,1 + dsine / 2, dsine)
lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat, flux_fdt[23])
plt.plot(lat, flux_hmi[23])

plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_24245/1444073679.py:1: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_hmi, axis=-1)
/tmp/ipykernel_24245/1444073679.py:2: RuntimeWarning: Mean of empty slice
  flux_fdt = np.nanmean(data_fdt, axis=-1)


In [52]:
data_pos = data_fdt[11].copy()
data_pos[data_pos < 0] = np.nan

data_neg = data_fdt[11].copy()
data_neg[data_neg > 0] = np.nan

flux_pos = np.nanmean(data_pos, axis=-1)
flux_neg = np.nanmean(data_neg, axis=-1)

dsine = 1 / 359.5
lat = np.arange(-1,1 + dsine / 2, dsine)
lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi


plt.figure(figsize=(10,8))
plt.plot(lat, flux_pos)
plt.plot(lat, flux_neg)

plt.xlim(-90,90)
plt.ylim(-20,20)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_24245/1380376451.py:7: RuntimeWarning: Mean of empty slice
  flux_pos = np.nanmean(data_pos, axis=-1)
/tmp/ipykernel_24245/1380376451.py:8: RuntimeWarning: Mean of empty slice
  flux_neg = np.nanmean(data_neg, axis=-1)


In [50]:
t = np.where(np.all([lat > 40, lat < 60], axis=0))

temp1 = data_fdt[23].copy()
temp1[temp1 > 0] = np.nan
temp1 = np.nanmean(temp1, axis=-1)

temp2 = data_hmi[23].copy()
temp2[temp2 > 0] = np.nan
temp2 = np.nanmean(temp2, axis=-1)

(np.trapezoid(temp1[t] * np.cos(lat[t] * np.pi / 180), x=lat[t]),
 np.trapezoid(temp2[t] * np.cos(lat[t] * np.pi / 180), x=lat[t]))

/tmp/ipykernel_24245/3189354363.py:13: RuntimeWarning: Mean of empty slice
  temp2 = np.nanmean(temp2, axis=-1)


(np.float64(-37.206948999559515), np.float64(-47.70788170378876))

In [55]:
t = np.where(np.all([lat > 60, lat < 80], axis=0))

temp1 = flux_fdt[23]
temp2 = flux_hmi[23]

(np.trapezoid(temp1[t] * np.cos(lat[t] * np.pi / 180), x=lat[t]),
 np.trapezoid(temp2[t] * np.cos(lat[t] * np.pi / 180), x=lat[t]))

(np.float64(-15.029687406901223), np.float64(-12.66698767278418))